# Cohort Design in OMOP

This notebook documents the cohort design step by step, following `../docs/protocol.md`. SQL is implemented in `../sql/02_cohort.sql` and `../sql/03_attrition.sql`, with a final cohort of 824 patients.

## 1. Clinical Question

We start from the question: among adults with cancer discharged from an inpatient hospitalization, can 180 days of prior clinical history improve prediction of 30-day readmission beyond prior hospital use?

## 2. OMOP Tables

The implemented SQL uses OMOP tables for patients, visits, observation periods, cancer diagnoses, and deaths. These support cohort eligibility and the temporal anchors for later readmission outcome and feature construction.

## 3. Adult Population

Patients must have reached their 18th birthday by `index_admission`. The SQL applies the exact birthday condition `CAST(birth_datetime AS DATE) <= visit_start_date - INTERVAL 18 YEAR`. This choice belongs in the cohort definition, not in model tuning.

## 4. Cancer Definition

The implemented cancer definition uses the existing set of 12 codes. The cancer diagnosis must occur before or on `index_admission`.

## 5. Eligible Inpatient Hospitalizations

This section will define which OMOP visits count as inpatient hospitalizations and which discharge records are eligible. The patient must be alive at discharge.

## 6. Index Admission and Discharge

The first eligible inpatient hospitalization defines two distinct anchors: `index_admission` is its start and `index_discharge` is its discharge. Each patient contributes one row. Features end before admission, while the readmission outcome begins after discharge.

## 7. Observation Window

Patients need at least 180 days of observable history before `index_admission` and at least 30 days of observation after `index_discharge`. The predictor window is `[index_admission - 180 days, index_admission)`.

## 8. 30-Day Readmission Outcome

The outcome is a new inpatient hospitalization after the index discharge and within the next 30 days. This section will keep the outcome definition separate from predictor construction.

## 9. Leakage Rules

Every longitudinal feature must satisfy `index_admission - 180 days <= feature_timestamp < index_admission`. No predictor may use information recorded at or after admission, including the index hospitalization. Outcome construction is separate and begins after `index_discharge`.

## 10. From Clinical Definition to SQL

The clinical rules are implemented in `../sql/02_cohort.sql`, and `../sql/03_attrition.sql` reports sequential eligibility counts. The final cohort and the final attrition step both contain 824 patients.